**Aharouni Refaël, Aye William (DIA1)** <br><br>
<h1 style="text-align:center; font-weight:bold;"> TD1 Web Datamining & Semantics</h1> <br><br><br>

## Lab 1 — Web Data Mining & Knowledge Extraction

### Objectives
The goal of this lab is to **automatically extract structured knowledge from unstructured web pages**.
We crawl Wikipedia articles about classical electromagnetism, apply Named Entity Recognition (NER)
to identify people, organizations, and locations, then extract Subject-Verb-Object triples to form
the seed of our Knowledge Graph.

### Why Electromagnetism?
We chose this domain because it has a **rich network of interconnected entities**:
scientists (Maxwell, Faraday, Tesla) are linked to each other through discoveries,
institutions (University of Cambridge, Royal Society), and awards (Nobel Prize, Copley Medal).
This density of relationships makes it ideal for building a meaningful knowledge graph.

### Pipeline Overview
```
Wikipedia articles  →  trafilatura (clean text)  →  SpaCy NER (entities)
                                                  →  Dependency parsing (SVO triples)
                                                  →  RDF export (initial_kb.ttl)
```

In [ ]:
import trafilatura
import pandas as pd
import spacy
import httpx

In [ ]:
nlp = spacy.load("en_core_web_trf")

### Seed URL Selection & Crawler Ethics

**Why these specific pages?** We organized our seed URLs into 4 categories to ensure
broad domain coverage:

| Category | Purpose | Examples |
|----------|---------|----------|
| Core concepts | Define the domain scope | Electromagnetism, Electric field, Magnetic field |
| Scientists | People with rich relationship networks | Maxwell, Faraday, Tesla, Ampere, Hertz |
| Physical laws | Connect scientists to their discoveries | Maxwell's equations, Lorentz force, Faraday's law |
| Institutions | Organizations linking multiple scientists | University of Cambridge, Royal Society |

**Crawler ethics:** We respect Wikipedia's `robots.txt` and use `trafilatura` which handles
rate limiting. We also set a User-Agent string identifying our bot.

In [ ]:
seed_urls = [
    "https://en.wikipedia.org/wiki/Electromagnetism",
    "https://en.wikipedia.org/wiki/Electric_field",
    "https://en.wikipedia.org/wiki/Magnetic_field",
    "https://en.wikipedia.org/wiki/Maxwell%27s_equations",
]

In [ ]:
scientists = [
    "https://en.wikipedia.org/wiki/James_Clerk_Maxwell",
    "https://en.wikipedia.org/wiki/Michael_Faraday",
    "https://en.wikipedia.org/wiki/Nikola_Tesla",
    "https://en.wikipedia.org/wiki/Andr%C3%A9-Marie_Amp%C3%A8re",
    "https://en.wikipedia.org/wiki/Hendrik_Lorentz",
    "https://en.wikipedia.org/wiki/Heinrich_Hertz",
    "https://en.wikipedia.org/wiki/Oliver_Heaviside",
    "https://en.wikipedia.org/wiki/John_Ambrose_Fleming",
]

In [ ]:
concepts = [
    "https://en.wikipedia.org/wiki/Electromagnetic_induction",
    "https://en.wikipedia.org/wiki/Lorentz_force",
    "https://en.wikipedia.org/wiki/Electric_charge",
    "https://en.wikipedia.org/wiki/Electromagnetic_wave",
    "https://en.wikipedia.org/wiki/Electrostatics",
    "https://en.wikipedia.org/wiki/Gauss%27s_law",
    "https://en.wikipedia.org/wiki/Amp%C3%A8re%27s_law",
    "https://en.wikipedia.org/wiki/Faraday%27s_law_of_induction",
]

In [ ]:
laws = [
    "https://en.wikipedia.org/wiki/Lorentz_force",
    "https://en.wikipedia.org/wiki/Ohm%27s_law",
]

In [ ]:
institutions = [
    "https://en.wikipedia.org/wiki/University_of_Cambridge",
    "https://en.wikipedia.org/wiki/Royal_Society",
]

In [ ]:
context = [
    "https://en.wikipedia.org/wiki/Classical_electromagnetism",
    "https://en.wikipedia.org/wiki/Physics",
]

In [ ]:
extra = [
    "https://en.wikipedia.org/wiki/Classical_physics",
    "https://en.wikipedia.org/wiki/Physics",
]

In [ ]:
seed_urls = seed_urls + scientists + concepts + laws + institutions + context + extra

### Crawling & Cleaning Pipeline

**Why trafilatura instead of BeautifulSoup?**

Wikipedia pages are full of noise: navigation bars, infoboxes, citation markers,
"See also" sections, and footer links. Manually writing BeautifulSoup selectors to
strip all of this is fragile and error-prone.

`trafilatura` is specifically designed for **main content extraction** — it uses
heuristics and machine learning to identify the main article text and discard
boilerplate. This gives us clean, paragraph-level text ready for NER.

**Cleaning steps:**
1. `trafilatura.fetch_url()` downloads the raw HTML
2. `trafilatura.extract()` strips navigation, ads, and metadata
3. The result is clean text split into sentences by SpaCy

In [ ]:
def fetch_clean_text(url):
    try:
        text = trafilatura.extract(trafilatura.fetch_url(url))
        
        if text and len(text.split()) > 300:
            return text
        return None
    except:
        return None

### Named Entity Recognition (NER)

**Why SpaCy's `en_core_web_trf` (transformer model)?**

We chose the transformer-based model over the smaller statistical ones (`en_core_web_sm`/`lg`)
for three reasons:

1. **Contextual disambiguation**: 'Maxwell' can refer to a person (James Clerk Maxwell) or
   a concept (Maxwell's equations). The transformer understands context and resolves this.
2. **Technical vocabulary**: Physics text contains specialized terms that statistical models
   often misclassify (e.g., tagging 'Lorentz force' as PERSON instead of a concept).
3. **Higher F1 score**: `en_core_web_trf` achieves ~90% F1 on OntoNotes NER vs ~85% for `lg`.

**Entity types we extract:**
- **PERSON** — Scientists, researchers (Maxwell, Faraday, Tesla)
- **ORG** — Institutions, societies (Royal Society, IEEE)
- **GPE** — Countries, cities (Scotland, France, Edinburgh)
- **LOC** — Geographic locations

In [ ]:
def extract_entities(text):
    doc = nlp(text)
    
    entities = []
    
    for ent in doc.ents:
        if ent.label_ in ["PERSON", "ORG", "GPE", "LOC", "DATE"]:
            entities.append((ent.text.strip(), ent.label_))
    
    return list(set(entities))

In [ ]:
def extract_relations(text):
    doc = nlp(text)
    triples = []
    
    for sent in doc.sents:
        ents = [ent for ent in sent.ents if ent.label_ in ["PERSON","ORG","GPE"]]
        
        if len(ents) < 2:
            continue
        
        verbs = [token for token in sent if token.pos_ == "VERB"]
        
        # Try dependency-based extraction first
        found = False
        for token in verbs:
            subjects = [w for w in token.lefts if w.dep_ in ("nsubj","nsubjpass")]
            objects = [w for w in token.rights if w.dep_ in ("dobj","pobj")]
            
            if subjects and objects:
                subj = subjects[0].text
                obj = objects[0].text
                rel = token.lemma_
                
                triples.append((subj, rel, obj))
                found = True
        
        # Fallback: entity co-occurrence + verb
        if not found and verbs:
            rel = verbs[0].lemma_
            for i in range(len(ents) - 1):
                triples.append((ents[i].text, rel, ents[i+1].text))
    
    return triples

### Relation Extraction

**Why SVO (Subject-Verb-Object) triples?**

Knowledge graphs are built from triples: `(subject, predicate, object)`. The simplest way
to extract these from text is **dependency parsing**: SpaCy identifies the syntactic structure
of each sentence, and we extract the subject (nsubj), the verb (ROOT), and the direct object (dobj).

Examples from our corpus:
- *(James Clerk Maxwell, formulated, Maxwell's equations)*
- *(Michael Faraday, discovered, electromagnetic induction)*
- *(Heinrich Hertz, demonstrated, electromagnetic waves)*

**Limitations:** SVO extraction is noisy — complex sentences with relative clauses, passive voice,
or coordination produce incomplete or incorrect triples. That's why we later **link to Wikidata**
in Lab 4: Wikidata provides clean, curated relationships that complement our noisy NER output.

In [ ]:
all_triples = []
all_entities = []

for url in seed_urls:
    print("Processing:", url)
    
    text = fetch_clean_text(url)
    
    if text:
        entities = extract_entities(text)
        relations = extract_relations(text)
        
        all_entities.extend([(e[0], e[1], url) for e in entities])
        all_triples.extend([(h, r, t, url) for (h,r,t) in relations])

Processing: https://en.wikipedia.org/wiki/Electromagnetism
Processing: https://en.wikipedia.org/wiki/Electric_field
Processing: https://en.wikipedia.org/wiki/Magnetic_field
Processing: https://en.wikipedia.org/wiki/Maxwell%27s_equations
Processing: https://en.wikipedia.org/wiki/James_Clerk_Maxwell
Processing: https://en.wikipedia.org/wiki/Michael_Faraday
Processing: https://en.wikipedia.org/wiki/Nikola_Tesla
Processing: https://en.wikipedia.org/wiki/Andr%C3%A9-Marie_Amp%C3%A8re
Processing: https://en.wikipedia.org/wiki/Hendrik_Lorentz
Processing: https://en.wikipedia.org/wiki/Electromagnetic_induction
Processing: https://en.wikipedia.org/wiki/Lorentz_force
Processing: https://en.wikipedia.org/wiki/Electric_charge
Processing: https://en.wikipedia.org/wiki/Electromagnetic_wave
Processing: https://en.wikipedia.org/wiki/University_of_Cambridge
Processing: https://en.wikipedia.org/wiki/Royal_Society
Processing: https://en.wikipedia.org/wiki/Classical_physics
Processing: https://en.wikipedia

In [ ]:
df_entities = pd.DataFrame(all_entities, columns=["entity","type","source"])
df_triples = pd.DataFrame(all_triples, columns=["head","relation","tail","source"])

df_entities.to_csv("../data/extracted_knowledge.csv", index=False)
df_triples.to_csv("../data/triples.csv", index=False)

print("Saved files")

Saved files


### Initial Knowledge Graph (RDF Export)

**Why export to RDF (Turtle format)?**

RDF (Resource Description Framework) is the W3C standard for representing knowledge graphs.
By exporting our NER results as RDF triples, we make them:
- **Machine-readable** — any SPARQL engine can query them
- **Linkable** — entities can be connected to Wikidata URIs later
- **Interoperable** — follows the same format used by DBpedia, Wikidata, and other LOD datasets

We use `rdflib` to build the graph and serialize it as Turtle (`.ttl`), which is the most
human-readable RDF format.

In [ ]:
from rdflib import Graph, URIRef, Literal, Namespace
from rdflib.namespace import RDF, RDFS, OWL
import re, os

EX = Namespace("http://example.org/entity/")
REL = Namespace("http://example.org/relation/")

def to_uri_safe(s):
    """Convert an entity string to a URI-safe token."""
    return re.sub(r"[^A-Za-z0-9_]", "_", str(s).strip())[:80]

g = Graph()
g.bind("ex", EX)
g.bind("rel", REL)

# Add entity type triples
for _, row in df_entities.drop_duplicates("entity").iterrows():
    subj = EX[to_uri_safe(row["entity"])]
    g.add((subj, RDFS.label, Literal(str(row["entity"]))))
    g.add((subj, RDF.type, EX[to_uri_safe(row["type"])]))

# Add relation triples
for _, row in df_triples.drop_duplicates(["head","relation","tail"]).iterrows():
    subj = EX[to_uri_safe(row["head"])]
    pred = REL[to_uri_safe(row["relation"])]
    obj  = EX[to_uri_safe(row["tail"])]
    g.add((subj, pred, obj))

os.makedirs("../kg_artifacts", exist_ok=True)
g.serialize(destination="../kg_artifacts/initial_kb.ttl", format="turtle")
print(f"Initial KB: {len(g)} triples -> ../kg_artifacts/initial_kb.ttl")


### Ambiguity Cases & Entity Linking

NER is not perfect — here are **3 ambiguity cases** we encountered:

**1. "Maxwell" (PERSON vs. concept)**
The token 'Maxwell' appears both as a person name (James Clerk Maxwell) and as part of
'Maxwell's equations' (a physics concept). SpaCy sometimes tags the whole phrase as PERSON
due to the possessive form. We detect this by checking if the noun phrase ends with a concept
word (equations, law, force).

**2. "Cambridge" (GPE vs. ORG)**
'Cambridge' can refer to the city (GPE) or the University of Cambridge (ORG).
In 'studied at Cambridge' it is ORG; in 'born in Cambridge' it is GPE.
Both are contextually correct, showing why **entity linking** (Lab 4) is essential
— Wikidata assigns different QIDs to the city vs. the university.

**3. "Royal Society" (ORG vs. MISC)**
'Royal Society' was occasionally misclassified in possessive constructions like
'the Royal Society's proceedings'. This is a known weakness of span-based NER
models with possessive markers.

In [ ]:
df_entities

,entity,type,source
0,R.G. Lerner,PERSON,https://en.wikipedia.org/wiki/Electromagnetism
1,Georgia State University,ORG,https://en.wikipedia.org/wiki/Electromagnetism
2,Michael,PERSON,https://en.wikipedia.org/wiki/Electromagnetism
3,Fleisch,PERSON,https://en.wikipedia.org/wiki/Electromagnetism
4,W.H. Freeman,ORG,https://en.wikipedia.org/wiki/Electromagnetism
...,...,...,...
7062,Jim,PERSON,https://en.wikipedia.org/wiki/Physics
7063,Dover Publications,ORG,https://en.wikipedia.org/wiki/Physics
7064,Krzanowski,PERSON,https://en.wikipedia.org/wiki/Physics
7065,1993,DATE,https://en.wikipedia.org/wiki/Physics
